In [8]:
import numpy as np
x= np.arange(20) 
x[0] = 20
print(x)




[20  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


In [15]:
x = np.arange(10)
x[9] = 10
print(x)



[ 0  1  2  3  4  5  6  7  8 10]


In [6]:
import jax
import jax.numpy as jnp


In [27]:
x = jnp.arange(10)
y = x.at[0].set(10)


#print(x)
print(y)



[10  1  2  3  4  5  6  7  8  9]


Fardal: A different particle release method described in GD1_and_SubhaloFitting_GPU.ipynb 
where an execution strategy pot.gen_stream_vmapped(...) is used with seed_num (controls the randomnes)

Here we switch to a different generator ssc.gen_stream_Chen25(...) which is defined in streamhelpers.py it uses the JAX PRNGkey to generate random particles and vmap for vectorised mapping. 

In [ ]:
import streamsculptor as ssc #provide the Chen25 stream generation function (used for unperturbed stream), 
                             #function imported directly for the perturbed streams calc later


In [ ]:
1. a = [2, 3, 4], b = [1, 0, 5] einsum('i,i->', a, b) 
2+20=22

2. einsum('i,i->i',a,b) 
    [2,0,20]

3. M = [[1, 0], [2, 1]], v = [3, 4] einsum('ij,j->i', M, v)
for i=0, M[0,0]x3 + M[0,1]x4 = 1x3 +0x4 = 3
for i=1, M[1,0]x3 + M[1,1]x4 = 2x3 +1x4 = 10

[3,10]

In [12]:
a = [2, 3, 4]
b = [1, 0, 5]
result = jnp.einsum('i,i->', a, b)
print(result)

r2= jnp.einsum('i,i->i',a,b)
print(r2)

M = [[1, 0], [2, 1]]
v = [3, 4]
r3 = jnp.einsum('ij,j->i', M, v)
print(r3)

22
[ 2  0 20]
[ 3 10]


In [18]:
R = [[2, 0], [0, 3]]  
icrs_vec = [[1, 1], [2, 0]]
rotated = jnp.einsum('ij,kj->ik', R, icrs_vec) 
print(rotated)
"""
Star 0: 2x1 + 1x0 = 2
        0x1 + 3x1 = 3
[2,3]
Star 1: 2x2 + 0x0 = 4
        0x2 + 3x0 = 0
[4,0] 
"""
#ki one row per star
#ik one row per component 

[[2 4]
 [3 0]]


'\nStar 0: 2x1 + 1x0 = 2\n        0x1 + 3x1 = 3\n[2,3]\nStar 1: 2x2 + 0x0 = 4\n        0x2 + 3x0 = 0\n[4,0] \n'

In [20]:
R = [[0,-1],[1,0]]
icrs_vec = [[2,0],[0,3],[1,1]]
r4= jnp.einsum('ij,kj->ki', R, icrs_vec)
print(r4)

# einsum would work in an identical way as icrs_vec @ R.T 
"""
[[0,2],[-3,0],[-1,1]]
"""

[[ 0  2]
 [-3  0]
 [-1  1]]


'\n[[0,2],[-3,0],[-1,1]]\n'

In [21]:
R = [[1,0,0],[0,0,-1],[0,1,0]]
icrs_vec = [[1,2,3]]
r5 = jnp.einsum('ij,kj->ki', R, icrs_vec)
print(r5)

[[ 1 -3  2]]


In [35]:

def icrs_to_gd1(ra,dec):
    R = jnp.array( [
            [-0.4776303088, -0.1738432154, 0.8611897727],
            [0.510844589, -0.8524449229, 0.111245042],
            [0.7147776536, 0.4930681392, 0.4959603976],
        ]
    )
    cart_coord = jnp.vstack([jnp.cos(ra)*jnp.cos(dec),
                         jnp.sin(ra)*jnp.cos(dec),
                         jnp.sin(dec)]).T
    stream_frame = jnp.einsum('ij,kj->ki',R,cart_coord)
    phi1 = jnp.arctan2(stream_frame[:,1],stream_frame[:,0])*180/jnp.pi
    phi2 = jnp.arcsin(stream_frame[:,2])*180/jnp.pi
    
    return phi1, phi2

ra = jnp.array([90*jnp.pi/180])
dec = jnp.array([0*jnp.pi/180])
phi1, phi2 = icrs_to_gd1(ra, dec)
print(phi1, phi2)



[-101.52654] [29.542439]


In [ ]:
def gen_stream_vmapped_Chen25(pot_base: callable, ts: jnp.array, prog_w0: jnp.array, Msat: float, key: jax.random.PRNGKey,
                              solver=diffrax.Dopri5(scan_kind='bounded'), rtol=1e-7, atol=1e-7, dtmin=0.3, dtmax=None, max_steps=10_000,
                              throw=False, prog_pot=potential.PlummerPotential(m=0.0, r_s=1.0,units=usys)):
    stream_ics, orb_fwd = gen_stream_ics_Chen25(pot_base=pot_base, ts=ts, prog_w0=prog_w0, Msat=Msat, key=key, 
                                                solver=solver, rtol=rtol, atol=atol, dtmin=dtmin, dtmax=dtmax, max_steps=max_steps)
    pos_close_arr, pos_far_arr, vel_close_arr, vel_far_arr = stream_ics
def gen_stream_Chen25(pot_base, ts, prog_w0, Msat, key, solver=diffrax.Dopri8(),
                      rtol=1e-7, atol=1e-7, dtmin=0.3, dtmax=None, max_steps=10_000, K=120,
                      prog_pot=potential.PlummerPotential(m=0.0, r_s=1.0, units=usys), throw=False,
                      method="auto", pot_pert=None):